# Whole proteome analysis

TODO - verify this runs with main env

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pylab as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path    
import gseapy as gp
from gseapy import Msigdb
import os
from functools import reduce
import json
import math
import collections
import itertools
import requests
import ast
from matplotlib_venn import venn2, venn2_circles, venn3_circles, venn3
from wp_funcs import * 
import polars as pl

print(os.getcwd())
%load_ext autoreload
%autoreload 2




#  When analyzing a new proteomic dataset, you should make a new Gene ID mapping.
input_folder_path = Path('/Users/henrysanford/dev/test_data/macrophage/01_whole_proteome/03_results/3_reps')
conditions_list = ["TLR1-2", "STING", "TLR3", "TLR4","TLR7","TLR8","TLR9"]
wp_file_path_all_channels_ratio  = input_folder_path / "03_combined_files" / '1' / "03_combfiles_forpca_channelratio_or_rawsignal_wp.csv"
output_path = input_folder_path / "04_results"

PERFORM_GO_TERM_ANALYSIS = False


In [ ]:
# remove proteins failing two replicate variability filter
wp_file_path_all_channels_ratio_df = pd.read_csv(wp_file_path_all_channels_ratio, index_col=[0])

filtered_out_path = input_folder_path / "03_combined_files" / '1'

df = wp_file_path_all_channels_ratio_df
wp_file_path_all_channels_ratio_df
rep_requirement = 2
pepnum_list = df.filter(regex="pepNum").columns.tolist()
df = df.drop(pepnum_list, axis=1)
# determine which uniprot + cysteine combos have fewer than X replicates
long_df = (df
        .drop(["protein", 
               "description"], axis=1)
        .melt(id_vars="uniprot")
        .dropna())

long_df[["condition","experiment","technical_replicate","drop"]] = long_df["variable"].str.split("_",n=3,expand=True)
long_df = (long_df.drop(["value","variable","technical_replicate","drop","condition"],axis=1)
        .drop_duplicates()
        .groupby("uniprot")
        .count()
        .reset_index())
passes_rep_filter_ids = set(long_df[long_df["experiment"] >= rep_requirement]["uniprot"])
wp_file_path_all_channels_ratio_df[~wp_file_path_all_channels_ratio_df["uniprot"].isin(passes_rep_filter_ids)].to_csv(filtered_out_path / "wp_fewer_than_2_replicates.csv")
# return df subset to residues passing requirement
wp_df = wp_file_path_all_channels_ratio_df[wp_file_path_all_channels_ratio_df["uniprot"].isin(passes_rep_filter_ids)]


# PCA

In [ ]:
pca_path = output_path / "pca_plots"
Path(pca_path).mkdir(exist_ok=True, parents=True)
pca_plot, loadings_df, pca_df, percent_df = get_pca_plot(wp_file_path_all_channels_ratio_df, "01_whole_proteome PCA", with_loadings = False)
loadings_df.to_csv(pca_path/ "loadings_df.csv")
pca_plot.write_html(pca_path / "pca.html")
pca_plot.show()

pca_plot_loadings, loadings_df, pca_df, percent_df = get_pca_plot(wp_file_path_all_channels_ratio_df, "01_whole_proteome PCA", with_loadings = True)
pca_df.to_csv(pca_path / "pca_results.csv")
# self-updating PC% axis labels for wp_visualization.Rmd
percent_df.to_csv(pca_path / "percent_explained.csv", index=False)
pca_plot_loadings.write_image(pca_path / "pca_loadings.svg", engine="kaleido")
pca_plot_loadings.write_html(pca_path / "pca_loadings.html")
loadings_df.to_csv(pca_path/ "loadings_df.csv")
pca_plot_loadings.show()

## GSEA of PC loadings

In [ ]:


msig = Msigdb()
# c5.go.bp = GO biological process
# c2.cp.reactome = reactome
gmt_name = "c5.go.bp"
# Pin to MSigDB 2026.1.Hs (same release as msigdbr / go_enrich). gseapy defaults to
# 2023.1.Hs, which still carries dropped terms like GOBP_NUCLEOTIDE_PHOSPHORYLATION.
gmt = msig.get_gmt(gmt_name, dbver="2026.1.Hs")



def perform_gsea_proteomics(loading_results, gene_sets, output_dir, pc):
    """
    Perform GSEA on proteomics loading results.
    """
    # Sort proteins by loading score
    ranked_proteins = loading_results.sort_values(ascending=True, by=pc)

    # Perform GSEA
    gsea_results = gp.prerank(
        rnk=ranked_proteins,
        gene_sets=gene_sets,
        processes=4,
        correction="bh",
        permutation_num=1000,
        outdir=output_dir,
        seed=42,
        no_plot=True,
    )

    return gsea_results




for pc in ["PC1", "PC2"]:
    pc_path = pca_path / pc
    pc_path.mkdir(exist_ok = True)
    pre_res = perform_gsea_proteomics(
        pd.DataFrame(loadings_df[pc]), gene_sets=gmt, output_dir=pc_path, pc=pc
    )

    df = pre_res.res2d
    df["abs_nes"] = -abs(df["NES"])
    df = df[df["NOM p-val"] < 0.01]

    terms = pre_res.res2d.Term[:10]  # Top 5 enriched terms
    fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))

    # Dot plot
    gp.plot.dotplot(
        pre_res.res2d,
        title=f"Top enriched terms for {pc}",
        cmap="viridis",
        size=5,
        top_term=15,
        ax=axes,
        column="FDR q-val",
    )
    plt.tight_layout()
    plt.savefig(
        pc_path / "top_enriched_terms.svg", bbox_inches="tight"
    )
    plt.show()

In [ ]:
# Filter to minimum four technical replicates

unwanted = wp_file_path_all_channels_ratio_df.columns[wp_file_path_all_channels_ratio_df.columns.str.startswith('pepNum')]
wp_file_path_all_channels_ratio_df.drop(unwanted, axis=1, inplace=True)
t_df = wp_file_path_all_channels_ratio_df.reset_index().drop(["protein", "description"], axis=1).set_index("uniprot").T.reset_index()
t_df["group"] = t_df["index"].str.split("_processed").str[0].str.rsplit("_").str[0]
t_df = t_df.drop("index", axis=1)
t_df = t_df.groupby("group").count()
t_df = t_df.T.max(axis=1) 
t_df = t_df[t_df >= 4] # min four replicates
uniprot_list = t_df.index.tolist()
wp_file_path_all_channels_ratio_df = wp_file_path_all_channels_ratio_df.reset_index()[wp_file_path_all_channels_ratio_df.reset_index()["uniprot"].isin(uniprot_list)]
wp_file_path_all_channels_ratio_df.drop("index", axis=1, inplace=True)

# Volcano plot

In [ ]:
volcano_path = output_path / "volcano_plots"
Path(volcano_path).mkdir(exist_ok=True)

volcano_filename = volcano_path / "volcano_data.csv"
volcano_df, copy_df, long_volcano_df = get_volcano_plot_treatment_vs_control(
    conditions_list,
    "M0",
    wp_file_path_all_channels_ratio_df,
    "Macrophage WP",
    output_path,
)
print(volcano_df)
volcano_df.to_csv(output_path / "volcano_plots" / "volcano_data.csv")

## Curating gene sets

Substring searches to create curated biological pathway lists of proteins


In [ ]:
study_uniprots = set(volcano_df.reset_index()["uniprot"])


def map_genes_to_uniprot(df, gene_col_name="genes"):
    r = requests.post(
        url="https://biit.cs.ut.ee/gprofiler/api/convert/convert/",
        json={
            "organism": "hsapiens",
            "target": "UNIPROTSWISSPROT_ACC",
            "query": list(df["genes"]),
        },
    )
    r.json()["result"]
    gene_to_protein_dict = dict()
    for entry in r.json()["result"]:
        if (
            entry["incoming"] in gene_to_protein_dict
            and gene_to_protein_dict[entry["incoming"]] != entry["converted"]
        ):
            if gene_to_protein_dict[entry["incoming"]] in study_uniprots:
                continue
        gene_to_protein_dict[entry["incoming"]] = entry["converted"]
    df["uniprot"] = df["genes"].apply(lambda x: gene_to_protein_dict[x])
    return df


def query_gene_ontology(substrings, domains, output_path):
    return query_gene_ontology_msigdb(
        substrings,
        domains,
        output_path,
        map_genes_to_uniprot=map_genes_to_uniprot,
    )

In [ ]:
# Substring searches to find related GO-terms
# I switched to exact GO-ids for speed once I determined related terms
if False:
    go_term_list_dir = output_path / "GO_term_lists"
    go_term_list_dir.mkdir(exist_ok=True)

    query_gene_ontology(
        substrings=["Endoplasmic reticulum"],
        domains=["CC",],
        output_path=go_term_list_dir / "Endoplasmic reticulum",
    )

    go_term_list_dir = output_path / "GO_term_lists"
    go_term_list_dir.mkdir(exist_ok=True)

    # Electron transport chain
    df = query_gene_ontology(
        ["electron transport chain"],
        ["CC", "MF", "BP"],
        go_term_list_dir / "Electron transport chain",
    )

    # Endocytosis
    df = query_gene_ontology(
        ["Endocytosis"], ["BP", "MF"], go_term_list_dir / "Endocytosis"
    )

    # Glycolytic process
    df = query_gene_ontology(
        ["glycolytic"], ["BP", "MF"], go_term_list_dir / "Glycolytic process"
    )

    # GTPase activity
    df = query_gene_ontology(["GTPase"], ["BP", "MF"], go_term_list_dir / "GTPase activity")

    # Mitochondrial
    df = query_gene_ontology(
        ["mitochondri"], ["CC"], go_term_list_dir / "Mitochondrial"
    )

    # Mitochondrial translation
    df = query_gene_ontology(
        ["mitochondrial translation"],
        ["BP"],
        go_term_list_dir / "Mitochondrial translation",
    )

    # Vesicle-mediated transport
    df = query_gene_ontology(
        [
            "vesicle-mediated transport",
            "endosome transport",
            "Golgi vesicle",
            "vesicle tethering to Golgi",
            "retrograde transport, endosome to Golgi",
        ],
        ["CC", "BP", "MF"],
        go_term_list_dir / "Vesicle-mediated transport",
    )


In [ ]:
go_term_list_dir = output_path / "GO_term_lists_focused"
go_term_list_dir.mkdir(exist_ok=True)
default_domains = ["CC", "BP", "MF"]
search_strings = {
    "Electron transport chain": [["GO:0022900"], default_domains],
    "Mitochondrial": [[
        "GO:0005739",
        "GO:0031966",
        "GO:0005759",
        "GO:0098800",
        "GO:0005758",
        "GO:0044233",
        "GO:0005740",
        "GO:0042645",
        "GO:0098799",
        "GO:0005746",
    ], default_domains],
    "Mitochondrial translation": [["GO:0032543"], default_domains],
    "Vesicle-mediated transport": [[
        "GO:0060627",
        "GO:0048193",
        "GO:0006888",
        "GO:0042147",
    ], default_domains],
    "GTPase activity":[["GTPase"], ["BP", "MF"]]
}

gene_lists = dict()


for key, value in search_strings.items():
    list_dir = go_term_list_dir / key
    list_dir.mkdir(exist_ok=True)
    gene_lists[key] = query_gene_ontology(
        substrings=value[0],
        domains=value[1],
        output_path=list_dir,
    )

gene_dicts = dict()
for key, value in gene_lists.items():
    gene_dicts[key] = set(value["uniprot"])

# Glycolysis panel = HALLMARK_GLYCOLYSIS (MSigDB 2026.1.Hs), replacing the GO "glycolytic
# process" term. Same MSigDB release as the GSEA above; map gene symbols -> uniprot.
hallmark_glycolysis = Msigdb().get_gmt("h.all", dbver="2026.1.Hs")["HALLMARK_GLYCOLYSIS"]
glyc_uniprots = map_genes_to_uniprot(pd.DataFrame({"genes": hallmark_glycolysis}))["uniprot"]
gene_dicts["Glycolysis"] = set(glyc_uniprots) - {"None", None}  # drop g:Profiler non-matches


# annotating all of the protein expression information with functional lists
go_term_list_dir = output_path / "GO_term_lists_focused"
for key, value in gene_dicts.items():
    long_volcano_df[key] = long_volcano_df.reset_index()["uniprot"].isin(value).values
long_volcano_df.to_csv(output_path / "volcano_plots" / "volcano_data_long_format.csv")



# GO term analysis

In [ ]:
Path(input_folder_path / "04_results" / "gene_id_mapping").mkdir(exist_ok=True)
gene_id_mapping = input_folder_path / "04_results" / "gene_id_mapping" / "gProfiler_query_results.csv" 
gene_id_mapping_df = pd.read_csv(gene_id_mapping)
# gProfiler now uses the column names ("initial_alias","converted_alias","name","description","namespace") instead of 
# (uniprot,gene_id,name,description,namespace), so rename columns
gene_id_mapping_df = gene_id_mapping_df.rename(columns={
    "initial_alias":"uniprot",
    "converted_alias":"gene_id"
})
gene_id_mapping_df = gene_id_mapping_df[["uniprot", "gene_id"]].drop_duplicates()
gene_id_mapping_df = gene_id_mapping_df.groupby("uniprot").agg(list)

for col in gene_id_mapping_df.columns: 
     gene_id_mapping_df[col] = [','.join(map(str, l)) for l in gene_id_mapping_df[col]]
     gene_id_mapping_df[col] = gene_id_mapping_df[col].str.split(",").str[0]
     gene_id_mapping_df[col] = gene_id_mapping_df[col].replace("None", np.nan)


volcano_df_merged = volcano_df.join(gene_id_mapping_df)

wp_file_path_all_channels_ratio_df = wp_file_path_all_channels_ratio_df.merge(gene_id_mapping_df.reset_index(), on="uniprot", how="left")
gene_id_list = wp_file_path_all_channels_ratio_df["gene_id"].dropna().astype(float).dropna().astype(int).tolist()
wp_file_path_all_channels_ratio_df.to_csv(input_folder_path / "04_results" / "gene_id_mapping" / "wp_file_path_all_channels_ratio_df.csv")

In [ ]:

exp_ref_folder_path = output_path / "goterm_analysis" / 'experiment_specific_background'
Path(exp_ref_folder_path).mkdir(exist_ok=True, parents=True)

wp_file_path_all_channels_ratio_df = pd.read_csv(input_folder_path / "04_results" / "gene_id_mapping" / "wp_file_path_all_channels_ratio_df.csv", index_col=0).reset_index()



_vol = pl.read_csv(output_path / "volcano_plots" / "volcano_data.csv")
_reg_cols = [c for c in _vol.columns if c.startswith("Regulation_")]
_wp_background = _vol.filter(pl.col("protein").is_not_null())["protein"].unique().to_list()

_direction_regs = {
    "up": ["Significant Up"],
    "down": ["Significant Down"],
    "up_and_down": ["Significant Up", "Significant Down"],
}

_rows = []
for _changing, _regs in _direction_regs.items():
    for _rc in _reg_cols:
        _cond = _rc.split("Macrophage WP - ")[1].split(" vs. M0")[0]
        for _g in (
            _vol.filter(pl.col(_rc).is_in(_regs), pl.col("protein").is_not_null())["protein"]
            .unique()
            .to_list()
        ):
            _rows.append({"changing_genes_tested": _changing, "condition": _cond, "gene": _g})

pl.DataFrame(_rows, schema={"changing_genes_tested": pl.Utf8, "condition": pl.Utf8, "gene": pl.Utf8}).write_csv(
    exp_ref_folder_path / "wp_changing_genes.csv"
)
pl.DataFrame({"gene": _wp_background}).write_csv(exp_ref_folder_path / "wp_background.csv")


In [ ]:
direction = "down"

# wrangle go results into wide and long format results
result_dfs = []
wide_dfs = []
for condition in conditions_list:
    df = pd.read_csv(
        # which percentage of GO terms are mapping to NA
        exp_ref_folder_path / f"goterm_ref_list_{condition} vs. M0_{direction}_go_term_results.csv",
        index_col = 0
    )

    wide_df = df.copy().set_index(['GO', 'term','class'])
    wide_df.columns = condition + "_" + wide_df.columns
    wide_dfs.append(wide_df.reset_index())

    df["condition"] = condition
    result_dfs.append(df)

long_df = pd.concat(result_dfs)

wide_df = reduce(lambda left, right: pd.merge(left, right, on=['GO', 'term','class'], how='outer'), wide_dfs)

long_df.to_csv(
    exp_ref_folder_path / f"aggregated_long_goterm_ref_list_{direction}_go_term_results.csv"
)

wide_df.to_csv(
    exp_ref_folder_path / f"aggregated_wide_goterm_ref_list_{direction}_go_term_results.csv"
)

## Condition correlation plots

In [ ]:
wp_file_path_all_channels_ratio_df = wp_file_path_all_channels_ratio_df.set_index(["uniprot", "protein", "description", "gene_id"])


In [ ]:
cond_list = ["M0", "TLR1-2", "STING", "TLR3", "TLR4","TLR7","TLR8","TLR9"]
for cond_name in cond_list:
    filter_col = [col for col in wp_file_path_all_channels_ratio_df if col.startswith(cond_name)]
    wp_file_path_all_channels_ratio_df['median_'+cond_name] = wp_file_path_all_channels_ratio_df.loc[:, filter_col].median(axis = 1)
    wp_file_path_all_channels_ratio_df['mean_'+cond_name] = wp_file_path_all_channels_ratio_df.loc[:, filter_col].mean(axis = 1)
filter_col = [col for col in wp_file_path_all_channels_ratio_df if col.startswith("median_")]
median_conditions_df = wp_file_path_all_channels_ratio_df[filter_col]
filter_col = [col for col in wp_file_path_all_channels_ratio_df if col.startswith("mean_")]
mean_conditions_df = wp_file_path_all_channels_ratio_df[filter_col]
wp_file_path_all_channels_ratio_df = wp_file_path_all_channels_ratio_df.drop(filter_col, axis=1)
median_conditions_df.columns = median_conditions_df.columns.str.replace('median_', '')
mean_conditions_df.columns = mean_conditions_df.columns.str.replace('mean_', '')

In [ ]:

corr_folder = input_folder_path / "04_results" / "corr_plots"
Path(corr_folder).mkdir(exist_ok=True)
for combination in itertools.combinations(conditions_list, 2):
    corr_plot_wp(mean_conditions_df, combination[0],combination[1], "M0", corr_folder)

## WP vs RNA Seq

In [ ]:
from src.uniprot_utils import create_entry_cache, get_function, get_go_terms

cache_df = mean_conditions_df.reset_index()
cache_df = cache_df[~cache_df["uniprot"].str.contains("contaminant")]

cache = create_entry_cache(cache_df.reset_index())

In [ ]:
wp_rna_corr_folder = input_folder_path / "04_results" / "wp_vs_rnaseq"
rna_seq_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/RNA-seq/"
)
Path(wp_rna_corr_folder).mkdir(exist_ok=True)

rna_de_results_dir = rna_seq_dir / "RNA-seq_Beatrice/macrophage_analysis/DEG/"
merged_path = rna_seq_dir / "RNA-seq_Henry/merged_rna_and_protein_de"


for condition in conditions_list:
    if condition == "TLR1-2":
        rna_condition = "TLR1"
    else:
        rna_condition = condition
    file_path = rna_de_results_dir / "M0lowvs{}/DE.xlsx".format(rna_condition)

    rna_de_df = pd.read_excel(file_path, sheet_name=1).drop("description", axis=1)

    ls = [str(i) for i in range(1, 23)] + ["X", "Y"]
    rna_de_df = rna_de_df[rna_de_df["chr"].isin(ls)]

    subdf_list = [rna_de_df, mean_conditions_df[["M0", condition]].reset_index()]
    df_merged = reduce(
        lambda left, right: pd.merge(
            left, right, left_on="gene.symbol", right_on="protein", how="outer"
        ),
        subdf_list,
    )
    df_merged_uniprot = reduce(
        lambda left, right: pd.merge(
            left, right, left_on="uniprot.id", right_on="uniprot", how="outer"
        ),
        subdf_list,
    )
    df_merged_uniprot["uniprot_function"] = df_merged_uniprot["uniprot"].apply(
        get_function, cache=cache
    )
    df_merged_uniprot["uniprot_goterms"] = df_merged_uniprot["uniprot"].apply(
        get_go_terms, cache=cache
    )
    df_merged = pd.concat([df_merged, df_merged_uniprot]).drop_duplicates()

    merged_fn = merged_path / (file_path.stem + "M0vs{}".format(condition) + ".csv")
    df_merged.to_csv(merged_fn)
    corr_plot_wp_rnaseq(merged_fn, condition, "M0", wp_rna_corr_folder)

In [ ]:
wp_rna_path = Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 2_Unenriched proteomics analysis/Panels/wp_vs_rnaseq')

for f in os.listdir(wp_rna_path):
    if ("csv" not in f) or ("gene" in f):
        continue
    print(f)

    df = pd.read_csv(wp_rna_path / f)

    df["uniprot_function"] = df["uniprot"].apply(
        get_function, cache=cache
    )
    df["uniprot_goterms"] = df["uniprot"].apply(
        get_go_terms, cache=cache
    )

    new_file_name = f.split(".csv")[0] + "_annotated" + ".csv"

    df.to_csv(
        wp_rna_path / new_file_name
    )

    

In [ ]:
wp_rna_path = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 2_Unenriched proteomics analysis/Panels/wp_vs_rnaseq"
)

for f in os.listdir(wp_rna_path):
    if "csv" not in f:
        continue
    if "annotated" not in f:
        continue


    df = pl.read_csv(wp_rna_path / f)
    print(df.columns)
    df = df.join(
        other=protein_half_lives.select("gene_name", "mean_monocyte_half_life"),
        left_on="gene.symbol",
        right_on="gene_name",
        how = "left"
    ).join(
        other = reported_complexes,
        on = "uniprot",
        how = "left"
    )

    df.write_csv(wp_rna_path / f)

# CORRELATION GO


In [ ]:
import os
from pathlib import Path
import polars as pl

wp_rna_path = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 2_Unenriched proteomics analysis/Panels/wp_vs_rnaseq"
)

# --- Export RNA-vs-WP correlation quadrant gene sets for GO enrichment ---
# Enrichment runs in R (GO enrichment chunk in wp_visualization.Rmd, go_enrich, MSigDB 2026 GO).
# Background is experiment-specific: every gene detected in that comparison (preserves the
# original goatools reference_list logic), expressed as gene symbols. Selection stays here.
_targets = []
_background = []
for f in os.listdir(wp_rna_path):
    if "csv" not in f or "combined" in f or "gene_list" in f:
        continue
    cond = f.split(" vs. ")[0]
    df = pl.read_csv(wp_rna_path / f)

    for _g in df.filter(pl.col("protein").is_not_null())["protein"].unique().to_list():
        _background.append({"condition": cond, "gene": _g})

    for direction in ["Up", "Down", "Both"]:
        if direction == "Up":
            direction_filter = pl.col("logFC_wp") > 0
        elif direction == "Down":
            direction_filter = pl.col("logFC_wp") < 0
        else:
            direction_filter = pl.col("logFC_wp").abs() > 0
        for quadrant in ["violet", "teal-yellow"]:
            test_genes = (
                df.filter(
                    pl.lit(quadrant).str.contains(pl.col("expression")),
                    pl.col("protein").is_not_null(),
                )
                .filter(direction_filter)["protein"]
                .unique()
                .to_list()
            )
            for _g in test_genes:
                _targets.append(
                    {"condition": cond, "quadrant": quadrant, "direction": direction, "gene": _g}
                )

pl.DataFrame(
    _targets,
    schema={"condition": pl.Utf8, "quadrant": pl.Utf8, "direction": pl.Utf8, "gene": pl.Utf8},
).write_csv("rna_vs_wp_go_targets.csv")
pl.DataFrame(_background, schema={"condition": pl.Utf8, "gene": pl.Utf8}).unique().write_csv(
    "rna_vs_wp_go_background.csv"
)

# after writing targets, run enrichment via R

In [ ]:
pl.DataFrame(
    _targets,
    schema={"condition": pl.Utf8, "quadrant": pl.Utf8, "direction": pl.Utf8, "gene": pl.Utf8},
)

In [ ]:
# Aggregation of the per-comparison GO results into rna_vs_wp_correlation_go_term_analysis_BP.csv
# now lives in R: the "Correlation GO dotplot" chunk of whole_proteome/wp_visualization.Rmd
# rebuilds it from the per-comparison CSVs so it always matches the current go_enrich() schema.


# Comparison to protein half life

Pull protein half-lives from [Kosinski et. al. protein turnover dataset](https://doi.org/10.1038/s41467-018-03106-1). 

They have info for several cell types but we only use Monocytes data. 

They have data in two biological replicates which are highly correlated. Use the mean half-life value for later analysis. If it's only quantified in one bio replicate, use that value.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns

half_life_columns = [
    "Monocytes replicate 1 half_life",
    "Monocytes replicate 2 half_life",
]

protein_half_lives = (
    pl.read_excel(
        "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/04_Relevant Papers/Protein_half_lives/41467_2018_3106_MOESM5_ESM.xlsx"
    )
    .select(cs.contains("Monocytes"), "gene_name")
    .with_columns(
        pl.mean_horizontal(half_life_columns).alias("mean_monocyte_half_life")
    )
)

sns.scatterplot(
    data = protein_half_lives.drop_nulls(subset=cs.contains("half_life")),
    x = half_life_columns[0],
    y = half_life_columns[1],
    hue = "Monocytes replicate 2 dataQual"
)
plt.yscale('log')
plt.xscale('log')


In [ ]:
half_life_dir = Path("/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 2_Unenriched proteomics analysis/Panels/protein_half_life/")
protein_half_lives = protein_half_lives.drop_nulls(subset="mean_monocyte_half_life")
expression_data_columns = {
    "log2FoldChange": "log2FC_rna",
    "logFC_wp": "log2FC_wp",
    "gene.symbol": "gene_name",
    "uniprot": "uniprot",
}
tlr4_data = (
    pl.read_csv(wp_rna_path / "TLR1-2 vs. M0_annotated.csv")
    .select(expression_data_columns.keys())
    .rename(expression_data_columns)
)

sns.regplot(
    data=tlr4_data,
    y="log2FC_rna",
    x="log2FC_wp",
)
plt.show()

reported_complexes = pl.read_csv(
    half_life_dir / "reported_complexes_corum_complexpro.csv"
)

tlr4_data = (
    tlr4_data.join(other=protein_half_lives, on="gene_name", how="inner")
    .join(other=reported_complexes, on="uniprot", how="left")
    .with_columns(
        (
            pl.col("CORUM_name").is_not_null()
            | pl.col("ComplexPortal_name").is_not_null()
        ).alias("reported_in_complex")
    )
)

sns.scatterplot(
    data=tlr4_data, y="log2FC_rna", x="log2FC_wp", hue="mean_monocyte_half_life"
)

In [ ]:
from scipy import stats
import numpy as np

# calculate the regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    tlr4_data["log2FC_wp"], tlr4_data["log2FC_rna"]
)

tlr4_data = tlr4_data.with_columns(
    (slope * pl.col("log2FC_wp") + intercept).alias("predicted")
).with_columns(
    (pl.col("log2FC_rna") - pl.col("predicted")).alias("residual"),
    pl.col("mean_monocyte_half_life")
    .log(base=2)
    .alias("log10_mean_monocyte_half_life"),
)


sns.lmplot(data=tlr4_data, x="log10_mean_monocyte_half_life", y="residual")

sns.scatterplot(
    data=tlr4_data,
    x="log10_mean_monocyte_half_life",
    y="residual",
    hue="reported_in_complex",
)

px.scatter(
    tlr4_data,
    x="log10_mean_monocyte_half_life",
    y="residual",
    color="reported_in_complex",
)

tlr4_data.write_csv(half_life_dir / "tlr4_correlation_half_life.csv")

In [ ]:
sns.scatterplot(
    data = tlr4_data,
    y = "log2FC_rna",
    x = "log2FC_wp",
    hue = "log10_mean_monocyte_half_life"
)

## GSEA

In [ ]:
gsea_dir = output_path / "gsea"
"""
df = pd.read_csv(
    '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/01_unenriched proteomics/03_results/20250124_3reps/03_combined_files/3reps/03_combfiles_forpca_channelratio_or_rawsignal_wp.csv',
    index_col=0
)

# drop pepNum, description, and protein columns

df = (df
      .drop(df.filter(like = "pepNum").columns, axis = 1)
      .drop(["protein", "description"], axis = 1))
df["Name"] = df["uniprot"]
df.drop("uniprot", axis = 1).set_index("Name").to_csv(gsea_dir / "macrophage_protein_for_gsea.txt", sep = "\t")
"""

def map_genes_to_uniprot(genes, gene_col_name = "genes"):
    study_uniprots = set(volcano_df.reset_index("uniprot"))
    r = requests.post(
        url='https://biit.cs.ut.ee/gprofiler/api/convert/convert/',
        json={
            'organism':'hsapiens',
            'target':'UNIPROTSWISSPROT_ACC',
            'query':genes,
        }
        )
    r.json()['result']
    gene_to_protein_dict = dict()
    for entry in r.json()["result"]:
        if entry["incoming"] in gene_to_protein_dict and gene_to_protein_dict[entry["incoming"]] != entry["converted"]:
            if gene_to_protein_dict[entry["incoming"]] in study_uniprots:
                print(entry["incoming"])
                print(entry["converted"])
                print(gene_to_protein_dict[entry["incoming"]])
                print("--------------------")
                continue
        gene_to_protein_dict[entry["incoming"]] = entry["converted"]
    uniprots = [gene_to_protein_dict[x] for x in genes]
    return uniprots


gsea_db = dict()
with open(gsea_dir / "macrophage_database.gmt") as file:
    for l in file.readlines():
        l  = l.split("\t")
        name = l[0]
        genes = l[2:]
        uniprots = map_genes_to_uniprot(genes)
        gsea_db[name] = "\t".join(uniprots)
with open(gsea_dir / "macrophage_database_uniprots.gmt", "w") as file:
    for name, uniprots in gsea_db.items():
        file.write("\t".join([name, uniprots, "\n"]))